# Dimension Dealers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable


In [0]:

df_bronze = spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .option("header", "true")\
    .load("abfss://bronze@adlscarproject.dfs.core.windows.net/rawdata")

df_bronze.display()

In [0]:
df_silver = df_bronze\
    .withColumn('End_Date',lit(None).cast(StringType()))\
    .withColumn('isActive',lit(1))\
    .withColumn("TimeStamp", current_timestamp())\
    .withColumn('HashKey', md5(concat_ws('||', 'BranchName', 'DealerName')))\
    .select('Branch_ID', 'Dealer_ID', 'BranchName', 'DealerName','isActive','End_Date','TimeStamp','HashKey')\
    .dropDuplicates(['Branch_ID','Dealer_ID','BranchName','DealerName'])\
    .sort('Branch_ID','Dealer_ID','BranchName','DealerName')
    
df_bronze.select('Date_id','Day','Month','Year').display()


Databricks data profile. Run in Databricks to view.

In [0]:
# Define Table
silver_table_name = 'Car_Project.Silver.Dealers'

# Create table if it doesn't exist
if not spark.catalog.tableExists(silver_table_name):
    spark.sql(f"""
                CREATE TABLE Car_Project.Silver.Dealers (
                    Dealer_Key BIGINT GENERATED ALWAYS AS IDENTITY, -- auto-increment surrogate key
                    Branch_ID STRING NOT NULL,
                    Dealer_ID STRING NOT NULL,
                    BranchName STRING,
                    DealerName STRING,
                    isActive INT,
                    End_Date TIMESTAMP,
                    TimeStamp TIMESTAMP,
                    HashKey STRING,
                    CONSTRAINT PK_BranchDealer PRIMARY KEY (Branch_ID, Dealer_ID)
                )
                USING DELTA
              """)
    print("Table created.")
else:
    print("Table already exists.")

In [0]:
# Separate active records only for merging
df_source = df_silver.filter(df_silver.isActive == 1)

# Load existing delta table (destination table) to compare with the incomind data
delta_dest = DeltaTable.forName(spark, silver_table_name)

# MERGE 1: Set old records records to inactive when a change is detected
delta_dest.alias("target").merge(
    df_source.alias("source"),
    "target.Branch_ID     = source.Branch_ID \
    and target.Dealer_ID  = source.Dealer_ID \
    and target.isActive   = 1 \
    and target.HashKey   != source.HashKey"
).whenMatchedUpdate(set={
    "End_Date": current_timestamp(),
    "isActive": lit(0)
}).execute()

# MERGE 2: Insert new versions of changed or new rows
delta_dest.alias("target").merge(
    df_source.alias("source"),
    "target.Branch_ID    = source.Branch_ID \
    AND target.Dealer_ID = source.Dealer_ID"
).whenNotMatchedInsert(values={
    "Branch_ID": "source.Branch_ID",
    "Dealer_ID": "source.Dealer_ID",
    "BranchName": "source.BranchName",
    "DealerName": "source.DealerName",
    "isActive": "1",
    "End_Date": "NULL",
    "TimeStamp": "source.TimeStamp",
    "HashKey": "source.HashKey"
}).execute()


In [0]:
%sql
-- DROP TABLE Car_Project.Silver.Dealers
select * from Car_Project.Silver.Dealers